# 🩺 01f — Pipeline densité médicale (DREES RPPS)

Construit `dim_medecins.parquet` (dept × année). Lancer `00_config_commun.ipynb`
avant. Code repris et adapté du notebook d'une collègue (`donnees_geographiques.ipynb`).

Source : DREES, "La démographie des professionnels de santé depuis 2012".
https://data.drees.solidarites-sante.gouv.fr/explore/dataset/la-demographie-des-professionnels-de-sante-de-2012-a-2024/
Fichier : `data/raw/demographie_medecins/Medecins_RPPS_2012_2026.xlsx`, feuille "Densités".

Contrairement à `dim_geo_pop`/`dim_csp` (statiques, 1 valeur figée par
département), cette table varie **par année** (2012-2026) — même logique
que la remarque faite en comparant les deux pipelines : la densité médicale
évolue réellement d'une année sur l'autre, pas de raison de la figer à la
dernière valeur connue partout. Clé de jointure : `dept` + `annee`.

Colonne produite : `densite_medecins` (médecins /100k hab, tous exercices,
tous âges, tous sexes, France métropolitaine).

In [ ]:
# Préambule : on se place dans le répertoire racine du projet et on ajoute le répertoire courant au PYTHONPATH pour pouvoir importer src/config.py

# pour recharger automatiquement les modules modifiés （src config surtout） sans redémarrer le kernel
%load_ext autoreload 
%autoreload 2

import os
import sys
from pathlib import Path
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, str(Path.cwd()))

import numpy as np
import pandas as pd
from src.config import RAW_DIR, TABLES_DIR, ANNEE_DEBUT, ANNEE_FIN, DEPTS,  DEPT_NOM_TO_CODE
from src.validation import valider_dim_table

print(
    f"Config chargée depuis src/config.py : {len(DEPTS)} départements | {ANNEE_DEBUT}–{ANNEE_FIN}")
print(f"RAW_DIR    = {RAW_DIR}")
print(f"TABLES_DIR = {TABLES_DIR}")

In [ ]:
def build_dim_medecins() -> pd.DataFrame:
    """
    Construit dim_medecins à partir du fichier DREES RPPS (feuille "Densités").

    CLÉ PRIMAIRE : dept × annee
    COLONNE : densite_medecins (médecins /100k hab)
    """
    fpath = RAW_DIR / "demographie_medecins" / "Medecins_RPPS_2012_2026.xlsx"
    if not fpath.exists(): # check si le fichier est présent, sinon on retourne un df vide
        print(f"⚠️  Fichier manquant : {fpath}")
        return pd.DataFrame()

    df = pd.read_excel(fpath, sheet_name="Densités")
    print(f"  Brut : {len(df):,} lignes")

    # France métropolitaine, tous médecins confondus, tous âges/sexes/modes d'exercice
    df = df[
        (df["territoire"] == "1-France métropolitaine")
        & (df["departement"] != "000-Ensemble")
        & (df["specialites_agregees"] == "00-Ensemble")
        & (df["exercice"] == "0-Ensemble")
        & (df["sexe"] == "0-Ensemble")
        & (df["tranche_age"] == "00-Ensemble")
    ].copy()
    print(f"  Après filtre France métro / tous médecins : {len(df):,} lignes")

    # Colonnes densite_YYYY -> passage au format long (1 ligne par dept x annee)
    cols_densite = [c for c in df.columns if c.startswith("densite_")]
    df = df[["departement"] + cols_densite].melt(
        id_vars=["departement"], value_vars=cols_densite,
        var_name="annee", value_name="densite_medecins"
    )
    df["annee"] = df["annee"].str[-4:].astype(int)

    # "075 -Paris" -> dept "75" (le code peut faire 2 ou 3 caractères : 075, 02A, 971...
    # les DOM sont déjà exclus par le filtre territoire ci-dessus)
    df[["dept", "dept_nom"]] = df["departement"].str.split(r"\s* -\s*", expand=True, n=1)
    df["dept"] = df["dept"].str.strip().str[-2:].str.upper()
    df = df[df["dept"].isin(DEPTS)]

    df = df[(df["annee"] >= ANNEE_DEBUT) & (df["annee"] <= ANNEE_FIN)]
    df["densite_medecins"] = pd.to_numeric(df["densite_medecins"], errors="coerce")

    df = df[["dept", "annee", "densite_medecins"]].reset_index(drop=True)
    print(f"  ✅ dim_medecins : {df.shape[0]:,} lignes × {df.shape[1]} colonnes")
    print(f"  Départements : {df['dept'].nunique()} | Années : {sorted(df['annee'].unique())}")
    return df


# ══════════════════════════════════════════════════════════════════════════════
# EXÉCUTION
# ══════════════════════════════════════════════════════════════════════════════
dim_medecins = build_dim_medecins()

if not dim_medecins.empty:
    valider_dim_table(dim_medecins, "dim_medecins", cle=["dept", "annee"])
    dim_medecins.to_parquet(TABLES_DIR / "dim_medecins.parquet", index=False)
    print(f"\n✅ Sauvegardé → data/processed/dim_medecins.parquet")
    display(dim_medecins.head(10))

  Brut : 301,320 lignes
  Après filtre France métro / tous médecins : 96 lignes
  ✅ dim_medecins : 576 lignes × 3 colonnes
  Départements : 96 | Années : [np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
── Validation de dim_medecins ──
  ✅ Tous les codes dept sont valides (96 départements)
  ✅ Aucun doublon sur la clé ['dept', 'annee']
  ✅ OK — prêt pour la fusion (dim_medecins)


✅ Sauvegardé → data/processed/dim_medecins.parquet


,dept,annee,densite_medecins
0,75,2020,884.85
1,77,2020,202.43
2,78,2020,278.26
3,91,2020,241.63
4,92,2020,400.69
5,93,2020,251.72
6,94,2020,376.57
7,95,2020,233.82
8,18,2020,212.01
9,28,2020,200.26


In [4]:
dim_medecins.head(10)

,dept,annee,densite_medecins
0,75,2020,884.85
1,77,2020,202.43
2,78,2020,278.26
3,91,2020,241.63
4,92,2020,400.69
5,93,2020,251.72
6,94,2020,376.57
7,95,2020,233.82
8,18,2020,212.01
9,28,2020,200.26
